In [9]:
import numpy as np
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.layers import (Input,Embedding,LSTM,Bidirectional,Dense,Dropout,Concatenate)

from tensorflow.keras.models import Model

from tensorflow.keras.callbacks import EarlyStopping

In [10]:
import pandas as pd

df=pd.read_csv("../data/processed/final_features.csv")

In [11]:
df.head()

,id,qid1,qid2,question1,question2,is_duplicate,q1_len,q2_len,q1_words,q2_words,common_words,total_unique_words,word_share,len_diff,fuzz_ratio,partial_ratio,tfidf_cosine_similarity
0,281827,401637,401638,the twin adarsh and anupam were born in may bu...,twin adarsh and anupam were born in may but th...,1,98,95,18,17,14,17,0.823529,3,98,100,1.000000
1,119174,193562,193563,whi cant i bare to watch someon be humili irlm...,how can you prove mean lifetim formula in radi...,0,129,61,22,10,0,31,0.000000,68,10,32,0.000000
2,18341,34756,2929,my question havent changed whi are they now be...,what happen to a question on quora if it is ma...,1,83,83,13,15,4,23,0.173913,0,53,59,0.481669
3,259132,374863,374864,can anyon tell the program for sequenc 4612183...,what are the next three number in thi series,0,67,47,8,9,1,16,0.062500,20,33,34,0.000000
4,208109,312009,201015,if the civil war occur today who would win,if there wa anoth us civil war north vs south ...,1,46,78,9,16,6,19,0.315789,32,61,50,0.455399


# Seperate Features

In [22]:
num_cols = [
    'q1_len',
    'q2_len',
    'q1_words',
    'q2_words',
    'common_words',
    'total_unique_words',
    'word_share',
    'len_diff',
    'fuzz_ratio',
    'partial_ratio',
    'tfidf_cosine_similarity'
]

X_num = df[num_features]

### prepares text data



In [14]:
tokenizer=Tokenizer()

all_questions=list(df['question1'].astype(str))+list(df['question2'].astype(str))

tokenizer.fit_on_texts(all_questions)

In [15]:
#vocab size

vocab_size=len(tokenizer.word_index)+1

print(vocab_size)

40174


In [18]:
#Convert text to sequneces

q1_seq=tokenizer.texts_to_sequences(
    df['question1'].astype(str)
)

q2_seq=tokenizer.texts_to_sequences(
    df['question2'].astype(str)
)

## padding

In [19]:
max_len=50

In [20]:
X_q1=pad_sequences(q1_seq,maxlen=max_len,padding='post')

X_q2=pad_sequences(q2_seq,maxlen=max_len,padding='post')

In [23]:
# numerical_featurss

X_num=df[num_cols]

In [24]:
#scale numerical features

scaler=StandardScaler()

X_num=scaler.fit_transform(X_num)

In [25]:
with open('../models/scaler.pkl','wb') as f:
    pickle.dump(scaler,f)

In [26]:
#target_varible

y=df['is_duplicate']

In [27]:
(X_q1_train,X_q1_test,X_q2_train,X_q2_test,X_num_train,X_num_test,y_train,y_test)=train_test_split(
    X_q1,X_q2,X_num,y,random_state=42,test_size=0.2
)

# Build Hybrid Model

In [28]:
input_q1=Input(shape=(max_len,))
input_q2=Input(shape=(max_len,))

In [29]:
# Embedding Layer

embedding=Embedding(input_dim=vocab_size,output_dim=100)

In [30]:
q1_emb=embedding(input_q1)
q2_emb=embedding(input_q2)



In [31]:
shared_bilstm=Bidirectional(LSTM(64,dropout=0.2))

In [33]:
q1_vec=shared_bilstm(q1_emb)
q2_vec=shared_bilstm(q2_emb)

In [34]:
#merge both questions

text_features=Concatenate()([q1_vec,q2_vec])

# Numerical feature branch

In [35]:
num_input=Input(shape=(X_num_train.shape[1],))

In [36]:
num_branch=Dense(32,activation='relu')(num_input)

num_branch=Dropout(0.2)(num_branch)

# merge text+ numerical features

In [37]:
merged=Concatenate()([text_features,num_branch])

# Fully conceted layer


In [45]:
x=Dense(128,activation='relu')(merged)
x=Dropout(0.3)(x)

x=Dense(64,activation='relu')(x)

output=Dense(1,activation='sigmoid')(x)


In [46]:
#Final model
model=Model(inputs=[input_q1,input_q2,num_input],outputs=output)

In [47]:
#compile

model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])

In [48]:
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 50)                │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ input_layer_1 (InputLayer)    │ (None, 50)                │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ embedding (Embedding)         │ (None, 50, 100)           │       4,017,400 │ input_layer[0][0],         │
│                               │                           │                 │ input_layer_1[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ input_layer_2 (InputLayer)    │ (None, 11)                │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ bidirectional (Bidirectional) │ (None, 128)               │          84,480 │ embedding[0][0],           │
│                               │                           │                 │ embedding[1][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense (Dense)                 │ (None, 32)                │             384 │ input_layer_2[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ concatenate (Concatenate)     │ (None, 256)               │               0 │ bidirectional[1][0],       │
│                               │                           │                 │ bidirectional[2][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout (Dropout)             │ (None, 32)                │               0 │ dense[0][0]                │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ concatenate_1 (Concatenate)   │ (None, 288)               │               0 │ concatenate[0][0],         │
│                               │                           │                 │ dropout[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_4 (Dense)               │ (None, 128)               │          36,992 │ concatenate_1[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_2 (Dropout)           │ (None, 128)               │               0 │ dense_4[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_5 (Dense)               │ (None, 64)                │           8,256 │ dropout_2[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_6 (Dense)               │ (None, 1)                 │              65 │ dense_5[0][0]              │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 4,147,577 (15.82 MB)

 Trainable params: 4,147,577 (15.82 MB)

 Non-trainable params: 0 (0.00 B)

In [50]:
#earlyStopping

early_stop=EarlyStopping(monitor='val_loss',patience=3,restore_best_weights=True)

In [51]:
history=model.fit([X_q1_train,X_q2_train,X_num_train],y_train,validation_data=([X_q1_test,X_q2_test,X_num_test],y_test),epochs=5,batch_size=128,callbacks=[early_stop])

Epoch 1/5
465/465 ━━━━━━━━━━━━━━━━━━━━ 112s 215ms/step - accuracy: 0.8531 - loss: 0.3358 - val_accuracy: 0.7889 - val_loss: 0.4609
Epoch 2/5
465/465 ━━━━━━━━━━━━━━━━━━━━ 94s 201ms/step - accuracy: 0.8711 - loss: 0.2992 - val_accuracy: 0.7815 - val_loss: 0.5092
Epoch 3/5
465/465 ━━━━━━━━━━━━━━━━━━━━ 94s 201ms/step - accuracy: 0.8923 - loss: 0.2534 - val_accuracy: 0.7873 - val_loss: 0.5260
Epoch 4/5
465/465 ━━━━━━━━━━━━━━━━━━━━ 95s 204ms/step - accuracy: 0.9077 - loss: 0.2176 - val_accuracy: 0.7803 - val_loss: 0.5948


In [52]:
loss,acc=model.evaluate(
    [X_q1_test,X_q2_test,X_num_test],y_test
)

print(acc)

465/465 ━━━━━━━━━━━━━━━━━━━━ 8s 17ms/step - accuracy: 0.7889 - loss: 0.4609
0.7888746857643127


In [53]:
y_pred_prob = model.predict(
    [
        X_q1_test,
        X_q2_test,
        X_num_test
    ]
)

465/465 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step


In [54]:
y_pred = (y_pred_prob > 0.5).astype(int)

In [55]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Precision: 0.7916556291390728
Recall: 0.792810717601804
F1 Score: 0.7922327523361389


In [57]:
results_df=pd.read_csv('model_results.csv')

In [58]:
results_df

,model,feature,accuracy,precision,recall,f1
0,Logistic Regression,TF-IDF,0.687117,0.663681,0.758621,0.707982
1,Random Forest,TF-IDF,0.743080,0.704847,0.836342,0.764985
2,XGB_BOOST,TF-IDF,0.752845,0.701136,0.881331,0.780974
3,SimpleRnn,Deep Learning Approach,0.522293,0.514534,0.789197,0.622933
4,LSTM,Deep Learning Approach,0.721175,0.741116,0.679822,0.709147
5,BiLSTM,Deep Learning Approach,0.722387,0.744955,0.676320,0.708981
6,Siamese,Dl approach,0.709725,0.694285,0.749461,0.720819


In [59]:
new_row=pd.DataFrame([{
    "model":"Hybrid lstm+Dense",
    "feature":"DL",
    "accuracy":acc,
    "precision":precision,
    "recall":recall,
    "f1":f1
}])

results_df=pd.concat(
    [results_df,new_row],
    ignore_index=True
)

In [60]:
results_df

,model,feature,accuracy,precision,recall,f1
0,Logistic Regression,TF-IDF,0.687117,0.663681,0.758621,0.707982
1,Random Forest,TF-IDF,0.743080,0.704847,0.836342,0.764985
2,XGB_BOOST,TF-IDF,0.752845,0.701136,0.881331,0.780974
3,SimpleRnn,Deep Learning Approach,0.522293,0.514534,0.789197,0.622933
4,LSTM,Deep Learning Approach,0.721175,0.741116,0.679822,0.709147
5,BiLSTM,Deep Learning Approach,0.722387,0.744955,0.676320,0.708981
6,Siamese,Dl approach,0.709725,0.694285,0.749461,0.720819
7,Hybrid lstm+Dense,DL,0.788875,0.791656,0.792811,0.792233


In [61]:
model.save('../models/hybrid_bilstm.keras')

In [62]:
with open('../models/tokenizer.pkl','wb') as f:
    pickle.dump(tokenizer,f)

In [63]:
with open('../models/scaler.pkl','wb') as f:
    pickle.dump(scaler,f)